# Xarray-Spatial Streams: D-inf and MFD stream ordering and link segmentation

Stream network analysis extracts channel topology from a DEM using flow routing models. xarray-spatial supports three routing variants: D8 (single steepest neighbor), D-infinity (continuous angle between two neighbors, Tarboton 1997), and MFD (flow fractions to all downslope neighbors). This notebook compares stream ordering and link segmentation across all three.

### What you'll build

1. Generate synthetic terrain and compute flow directions for D8, D-inf, and MFD
2. Compare Strahler stream ordering across all three routing models
3. Compare Shreve magnitude across routing models
4. Segment stream links and count network topology differences

![Stream analysis preview](images/stream_analysis_preview.png)

**Jump to a section:**
[Flow directions](#Flow-directions) | [Strahler ordering](#Strahler-ordering) | [Shreve magnitude](#Shreve-magnitude) | [Stream links](#Stream-links)

Standard imports plus flow direction, accumulation, stream ordering, and link functions for all three routing models.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from xrspatial import generate_terrain
from xrspatial import flow_direction, flow_direction_dinf, flow_direction_mfd
from xrspatial import flow_accumulation, flow_accumulation_mfd
from xrspatial import stream_order, stream_link
from xrspatial import stream_order_dinf, stream_link_dinf
from xrspatial import stream_order_mfd, stream_link_mfd

## Synthetic terrain

A 400x400 perlin-noise DEM with enough relief to produce a visible drainage network.

In [ ]:
W, H = 400, 400
template = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'],
                        coords={'y': np.linspace(0, 1000, H),
                                'x': np.linspace(0, 1000, W)})
terrain = generate_terrain(template, x_range=(0, 1000), y_range=(0, 1000))

fig, ax = plt.subplots(figsize=(8, 7))
terrain.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True,
                    cbar_kwargs={'label': 'Elevation'})
ax.set_title('Synthetic elevation')
ax.set_axis_off()
plt.tight_layout()

## Flow directions

Compute all three routing models from the same elevation surface. D8 picks one neighbor, D-inf distributes flow between two, and MFD distributes to all downslope neighbors.

In [ ]:
# D8
fd_d8 = flow_direction(terrain)
fa_d8 = flow_accumulation(fd_d8)

# D-infinity
fd_dinf = flow_direction_dinf(terrain)

# MFD
fd_mfd = flow_direction_mfd(terrain)
fa_mfd = flow_accumulation_mfd(fd_mfd)

print(f'D8 flow dir shape:   {fd_d8.shape}')
print(f'D-inf angles shape:  {fd_dinf.shape}')
print(f'MFD fractions shape: {fd_mfd.shape}')

## Strahler ordering

Strahler stream order assigns order 1 to headwater channels, and increments when two channels of equal order join. The plot compares network topology across all three routing models at the same accumulation threshold.

In [ ]:
threshold = 200

# D8 stream order
so_d8 = stream_order(fd_d8, fa_d8, threshold=threshold, method='strahler')

# D-inf stream order
so_dinf = stream_order_dinf(fd_dinf, fa_d8, threshold=threshold, method='strahler')

# MFD stream order
so_mfd = stream_order_mfd(fd_mfd, fa_mfd, threshold=threshold, method='strahler')

In [ ]:
vmax_strahler = max(float(np.nanmax(so_d8.values)),
                    float(np.nanmax(so_dinf.values)),
                    float(np.nanmax(so_mfd.values)))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, data, title in zip(axes, [so_d8, so_dinf, so_mfd], ['D8', 'D-infinity', 'MFD']):
    display = xr.where(data.isnull(), 0, data)
    display.plot.imshow(ax=ax, cmap='Blues', vmin=0, vmax=vmax_strahler,
                        add_colorbar=False)
    ax.set_title(f'Strahler order ({title})')
    ax.set_axis_off()

sm = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(0, vmax_strahler))
fig.colorbar(sm, ax=axes, shrink=0.6, label='Stream order')
plt.tight_layout()

# Save preview
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/stream_analysis_preview.png', bbox_inches='tight', dpi=120)

## Shreve magnitude

Shreve magnitude sums the count of upstream headwater channels at each point. Higher values indicate more upstream contributing area. The log scale makes the full range visible.

In [ ]:
sv_d8 = stream_order(fd_d8, fa_d8, threshold=threshold, method='shreve')
sv_dinf = stream_order_dinf(fd_dinf, fa_d8, threshold=threshold, method='shreve')
sv_mfd = stream_order_mfd(fd_mfd, fa_mfd, threshold=threshold, method='shreve')

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, data, title in zip(axes, [sv_d8, sv_dinf, sv_mfd], ['D8', 'D-infinity', 'MFD']):
    vals = xr.where(data.isnull(), 0, data)
    log_vals = np.log1p(vals)
    log_da = xr.DataArray(log_vals, dims=data.dims, coords=data.coords)
    log_da.plot.imshow(ax=ax, cmap='viridis', add_colorbar=False)
    ax.set_title(f'Shreve magnitude ({title})')
    ax.set_axis_off()

sm = plt.cm.ScalarMappable(cmap='viridis',
                           norm=plt.Normalize(0, float(np.log1p(np.nanmax(sv_d8.values)))))
fig.colorbar(sm, ax=axes, shrink=0.6, label='log(1 + magnitude)')
plt.tight_layout()

## Stream links

Link segmentation assigns a unique ID to each channel segment between junctions. MFD and D-inf can produce more junction points than D8 because flow splits across multiple neighbors.

In [ ]:
sl_d8 = stream_link(fd_d8, fa_d8, threshold=threshold)
sl_dinf = stream_link_dinf(fd_dinf, fa_d8, threshold=threshold)
sl_mfd = stream_link_mfd(fd_mfd, fa_mfd, threshold=threshold)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, data, title in zip(axes, [sl_d8, sl_dinf, sl_mfd], ['D8', 'D-infinity', 'MFD']):
    display = xr.where(data.isnull(), 0, np.mod(data, 20) + 1)
    display.plot.imshow(ax=ax, cmap='tab20', add_colorbar=False)
    ax.set_title(f'Stream links ({title})')
    ax.set_axis_off()

plt.tight_layout()

for label, data in [('D8', sl_d8), ('D-inf', sl_dinf), ('MFD', sl_mfd)]:
    n_links = len(np.unique(data.values[~np.isnan(data.values)]))
    n_stream = int(np.sum(~np.isnan(data.values)))
    print(f'{label:6s}: {n_links:4d} links, {n_stream:6d} stream cells')

<div class="alert alert-block alert-info">
<b>Choosing a routing model.</b> D8 is fastest and produces the sharpest single-path channels. D-inf gives smoother flow angles and is better for slope-area analysis. MFD distributes flow most realistically on flat terrain but produces wider, more diffuse channel networks. For stream ordering, D8 is usually sufficient. For detailed hillslope hydrology, MFD or D-inf are preferable.
</div>

### References

- Tarboton, D. G. (1997). [A new method for the determination of flow directions and upslope areas in grid digital elevation models](https://doi.org/10.1029/96WR03137). *Water Resources Research*, 33(2), 309-319.
- [Strahler stream order (Wikipedia)](https://en.wikipedia.org/wiki/Strahler_number)
- [xrspatial.stream_order API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.stream_order.html)